In [9]:
import pandas as pd

In [3]:
thoracic_tabsyn = pd.read_csv('synthetic/thoracic_surgery/tabsyn.csv')
thoracic_ganos = pd.read_csv('synthetic/thoracic_surgery/ganos.csv')
thoracic_codi = pd.read_csv('synthetic/thoracic_surgery/codi.csv')
thoracic_findiff = pd.read_csv('synthetic/thoracic_surgery/findiff.csv')
thoracic_great = pd.read_csv('synthetic/thoracic_surgery/great.csv')

models = ["tabsyn", "ganos", "codi"]

In [ ]:
from ctgan import CTGAN
import json

data_dir = f'data/thoracic_surgery'
info_path = f'{data_dir}/info.json'
train_dataset_path = f'{data_dir}/train.csv'

with open(info_path, 'r') as f:
    info = json.load(f)
    
train_df = pd.read_csv(train_dataset_path)

target_col_idx = info['target_col_idx'][0] if isinstance(info['target_col_idx'], list) else info['target_col_idx']
target_col = train_df.columns[target_col_idx]

num_cols = [train_df.columns[i] for i in info['num_col_idx']]
cat_cols = [train_df.columns[i] for i in info['cat_col_idx']]
cat_cols = [col for col in cat_cols if col != target_col]

# Initialize the CTGAN model
ctgan = CTGAN(
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    batch_size=500,  # Must be a multiple of `pac`
    epochs=300,
    pac=5  # Ensure `batch_size` is divisible by this value
)
ctgan.fit(train_df, discrete_columns=cat_cols)

# Generate synthetic data
synthetic_data = ctgan.sample(len(df))  # Generate the same number of rows as the original dataset

# Save the synthetic data to a CSV file (optional)
synthetic_data.to_csv('synthetic_data.csv', index=False)

# Print the first few rows of synthetic data
print(synthetic_data)


/home/pcrespo/miniconda3/envs/data_aug/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AssertionError: 

In [27]:
alpha_precissions = {}
beta_recalls = {}

thoracic_df = pd.DataFrame(columns=["Alpha Precission", "Beta Recall"])

dataname = "thoracic_tabsyn"

for model in models:
    quality_path = "eval/quality/thoracic_surgery/" + model + ".txt"
    with open(quality_path, "r") as f:
        lines = f.readlines()
        
        alpha_precission = float(lines[0].split(":")[1].strip())
        beta_recall = float(lines[1].split(":")[1].strip())
        
        alpha_precissions[model] = alpha_precission
        beta_recalls[model] = beta_recall
        

for model in models:
    thoracic_df.loc[model] = [alpha_precissions[model], beta_recalls[model]]

thoracic_df

,Alpha Precission,Beta Recall
tabsyn,0.950765,0.539837
ganos,0.895957,0.005041
codi,0.610407,0.440976
